# 📊 Training Diagnostics — HybridCNN

Análisis completo del historial de entrenamiento para identificar overfitting,
estabilidad del aprendizaje y señales de alarma.

> **Carga tu checkpoint** en la celda de configuración y ejecuta todo.


## 1 · Imports y configuración

In [ ]:
import pickle
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
from pathlib import Path

# ── Estilo ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#0f1117',
    'axes.facecolor':    '#161b27',
    'axes.edgecolor':    '#2a3045',
    'axes.labelcolor':   '#c8d0e0',
    'axes.titlecolor':   '#e8eaf0',
    'axes.grid':         True,
    'grid.color':        '#1e2535',
    'grid.linewidth':    0.8,
    'xtick.color':       '#7a8499',
    'ytick.color':       '#7a8499',
    'text.color':        '#c8d0e0',
    'legend.facecolor':  '#1a2030',
    'legend.edgecolor':  '#2a3045',
    'legend.fontsize':   9,
    'font.family':       'monospace',
    'lines.linewidth':   2.0,
})

C_TRAIN  = '#4fc3f7'   # azul claro
C_VAL    = '#f06292'   # rosa
C_TEST   = '#aed581'   # verde lima  (para la línea horizontal de test)
C_LR     = '#ffb74d'   # naranja
C_GAP    = '#ef9a9a'   # rojo suave
C_F1     = '#ce93d8'   # lila
C_PREC   = '#80cbc4'   # teal
C_RECALL = '#ffcc80'   # melocotón

print('✅ Imports OK')


## 2 · Cargar historial

In [ ]:
# ── Ajusta esta ruta al checkpoint que quieras analizar ──────────────────
from src.utils.config import CHECKPOINT_DIR

CHECKPOINT_PATH = CHECKPOINT_DIR / 'human_label_V2' / 'checkpoint_epoch_10.pt'

# ── Carga ────────────────────────────────────────────────────────────────────
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
H    = ckpt['history']          # dict con listas

# Métricas de test final (si las tienes, rellena manualmente)
TEST_ACC       = None   # ej: 0.82
TEST_LOSS      = None   # ej: 0.61
TEST_F1        = None   # ej: 0.80

epochs  = H['epoch']
best_epoch = ckpt.get('best_epoch', None)
best_acc   = ckpt.get('best_acc',   None)

print(f'📦 Checkpoint  : {CHECKPOINT_PATH.name}')
print(f'📌 Épocas       : {len(epochs)}')
print(f'🥇 Mejor epoch  : {best_epoch}  |  best_val_acc = {best_acc:.4f}')
print(f'🧪 Test acc     : {TEST_ACC}')
print()
print('Claves en history:', list(H.keys()))


## 3 · Resumen numérico rápido

In [ ]:
import pandas as pd

df = pd.DataFrame(H)

# Gap train-val
df['acc_gap']  = df['train_acc']  - df['val_acc']
df['loss_gap'] = df['val_loss']   - df['train_loss']

# Velocidad de mejora
df['val_acc_delta'] = df['val_acc'].diff().fillna(0)

print('📊 Últimas épocas:')
display(df[['epoch','train_loss','val_loss','train_acc','val_acc',
            'acc_gap','f1','lr']].tail(10).round(4))

print(f"\n🔴 Máx acc_gap  (train-val) : {df['acc_gap'].max():.4f}  (epoch {df.loc[df['acc_gap'].idxmax(),'epoch']})")
print(f"🔴 Máx loss_gap (val-train) : {df['loss_gap'].max():.4f}  (epoch {df.loc[df['loss_gap'].idxmax(),'epoch']})")
if TEST_ACC is not None:
    print(f"🔴 Gap val→test acc         : {best_acc - TEST_ACC:.4f}")
    print(f"   → {'⚠️  Overfitting sobre val' if best_acc - TEST_ACC > 0.04 else '✅ Gap val-test aceptable'}")


## 4 · Curvas de pérdida y accuracy

Las dos curvas más importantes: dónde divergen train y val.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Loss & Accuracy', fontsize=13, fontweight='bold', color='#e8eaf0', y=1.01)

# ── Loss ────────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs, H['train_loss'], color=C_TRAIN, label='Train Loss')
ax.plot(epochs, H['val_loss'],   color=C_VAL,   label='Val Loss',   linestyle='--')
if TEST_LOSS is not None:
    ax.axhline(TEST_LOSS, color=C_TEST, linestyle=':', linewidth=1.5, label=f'Test Loss={TEST_LOSS:.3f}')
if best_epoch:
    ax.axvline(best_epoch, color='#ffffff22', linestyle='--', linewidth=1)
    ax.text(best_epoch+0.1, ax.get_ylim()[1]*0.97, f'best\n(e{best_epoch})',
            fontsize=7, color='#ffffff55')
ax.set_title('Loss', fontsize=11)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend()

# ── Accuracy ────────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(epochs, H['train_acc'], color=C_TRAIN, label='Train Acc')
ax.plot(epochs, H['val_acc'],   color=C_VAL,   label='Val Acc',   linestyle='--')
if TEST_ACC is not None:
    ax.axhline(TEST_ACC, color=C_TEST, linestyle=':', linewidth=1.5, label=f'Test Acc={TEST_ACC:.3f}')
if best_epoch:
    ax.axvline(best_epoch, color='#ffffff22', linestyle='--', linewidth=1)
ax.set_title('Accuracy', fontsize=11)
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend()

plt.tight_layout()
plt.savefig('01_loss_acc.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()


## 5 · Gap de overfitting por época

El gap de accuracy (`train - val`) y de loss (`val - train`) revelan **cuándo**
empieza el overfitting. Una zona roja sostenida → el modelo memoriza.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

acc_gap  = [ta - va for ta, va in zip(H['train_acc'],  H['val_acc'])]
loss_gap = [vl - tl for vl, tl in zip(H['val_loss'],   H['train_loss'])]

# ── Acc gap ─────────────────────────────────────────────────────────────────
ax = axes[0]
ax.fill_between(epochs, acc_gap, alpha=0.25, color=C_GAP)
ax.plot(epochs, acc_gap, color=C_GAP, linewidth=1.5)
ax.axhline(0, color='#ffffff33', linewidth=0.8)
ax.axhline(0.05, color=C_GAP, linewidth=0.6, linestyle='--', alpha=0.5)
ax.text(epochs[-1], 0.051, ' umbral 5%', fontsize=7, color=C_GAP, va='bottom')
ax.set_title('Acc Gap  (train − val)', fontsize=11)
ax.set_xlabel('Epoch'); ax.set_ylabel('Gap')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

# ── Loss gap ─────────────────────────────────────────────────────────────────
ax = axes[1]
ax.fill_between(epochs, loss_gap, alpha=0.25, color=C_TRAIN)
ax.plot(epochs, loss_gap, color=C_TRAIN, linewidth=1.5)
ax.axhline(0, color='#ffffff33', linewidth=0.8)
ax.set_title('Loss Gap  (val − train)', fontsize=11)
ax.set_xlabel('Epoch'); ax.set_ylabel('Gap')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('02_overfitting_gap.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()


## 6 · Precision, Recall y F1

Si F1 diverge mucho de accuracy → clases desbalanceadas o el modelo aprende
a predecir clases mayoritarias.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(epochs, H['val_acc'],   color=C_VAL,    label='Val Acc',    linewidth=2)
ax.plot(epochs, H['f1'],        color=C_F1,     label='F1 (weighted)', linewidth=2)
ax.plot(epochs, H['precision'], color=C_PREC,   label='Precision',  linewidth=1.5, linestyle='--')
ax.plot(epochs, H['recall'],    color=C_RECALL, label='Recall',     linewidth=1.5, linestyle=':')

if TEST_F1 is not None:
    ax.axhline(TEST_F1, color=C_TEST, linestyle=':', linewidth=1.5, label=f'Test F1={TEST_F1:.3f}')

if best_epoch:
    ax.axvline(best_epoch, color='#ffffff22', linestyle='--', linewidth=1)

# Área de divergencia acc-f1
ax.fill_between(epochs, H['val_acc'], H['f1'], alpha=0.08,
                color=C_GAP, label='Δ acc-f1 (desbalance)')

ax.set_title('Métricas de validación (weighted)', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(ncol=3)

plt.tight_layout()
plt.savefig('03_metrics.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

# Diagnóstico automático
avg_gap = np.mean([a - f for a, f in zip(H['val_acc'], H['f1'])])
print(f'Δ medio acc-f1: {avg_gap:.4f}')
if avg_gap > 0.03:
    print('⚠️  Accuracy > F1 de forma sostenida → posible sesgo hacia clases mayoritarias')
else:
    print('✅ Acc y F1 alineados — no hay sesgo de clase grave')


## 7 · Learning Rate schedule

Cada bajada del LR debería ir seguida de una mejora en val_acc.
Si no lo hace → el modelo ya convergió o el LR inicial era demasiado alto.


In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4))

color_lr = C_LR
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Learning Rate', color=color_lr)
ax1.semilogy(epochs, H['lr'], color=color_lr, linewidth=2, label='LR')
ax1.tick_params(axis='y', labelcolor=color_lr)

ax2 = ax1.twinx()
ax2.set_ylabel('Val Acc', color=C_VAL)
ax2.plot(epochs, H['val_acc'], color=C_VAL, linestyle='--', linewidth=1.5, label='Val Acc')
ax2.tick_params(axis='y', labelcolor=C_VAL)
ax2.set_ylim(0, 1)

# Marcar bajadas de LR
lr_drops = [i for i in range(1, len(H['lr'])) if H['lr'][i] < H['lr'][i-1]]
for i in lr_drops:
    ax1.axvline(epochs[i], color='#ffffff22', linestyle=':', linewidth=1)
    ax1.text(epochs[i]+0.05, H['lr'][i]*1.2, f'↓LR', fontsize=7, color='#ffffff55')

ax1.set_title('Learning Rate vs Val Accuracy', fontsize=12)
ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('04_lr_schedule.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

print(f'LR drops en épocas: {[epochs[i] for i in lr_drops] if lr_drops else "ninguna"}')
print(f'LR inicial: {H["lr"][0]:.6f}  →  LR final: {H["lr"][-1]:.6f}')


## 8 · Velocidad de entrenamiento

Tiempo por época e imágenes/segundo. Picos → batches problemáticos o
contención de I/O. Útil para correlacionar con caídas de accuracy.


In [ ]:
has_time = 'epoch_time' in H and len(H['epoch_time']) == len(epochs)
has_ips  = 'images_per_sec' in H and len(H['images_per_sec']) == len(epochs)

if not has_time and not has_ips:
    print('⚠️  No hay datos de velocidad en este checkpoint.')
else:
    ncols = int(has_time) + int(has_ips)
    fig, axes = plt.subplots(1, ncols, figsize=(7*ncols, 4))
    if ncols == 1:
        axes = [axes]

    col = 0
    if has_time:
        ax = axes[col]; col += 1
        ax.bar(epochs, H['epoch_time'], color=C_TRAIN, alpha=0.7, width=0.6)
        ax.axhline(np.mean(H['epoch_time']), color=C_VAL, linestyle='--',
                   linewidth=1.2, label=f'Media {np.mean(H["epoch_time"]):.1f}s')
        ax.set_title('Tiempo por época (s)', fontsize=11)
        ax.set_xlabel('Epoch'); ax.set_ylabel('Segundos')
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.legend()

    if has_ips:
        ax = axes[col]
        ax.plot(epochs, H['images_per_sec'], color=C_LR, linewidth=2)
        ax.fill_between(epochs, H['images_per_sec'], alpha=0.15, color=C_LR)
        ax.axhline(np.mean(H['images_per_sec']), color='#ffffff55', linestyle='--',
                   linewidth=1, label=f'Media {np.mean(H["images_per_sec"]):.0f} img/s')
        ax.set_title('Throughput (img/s)', fontsize=11)
        ax.set_xlabel('Epoch'); ax.set_ylabel('img/s')
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.legend()

    plt.tight_layout()
    plt.savefig('05_speed.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()


## 9 · Dashboard resumen (una sola figura exportable)

Vista compacta con las 4 métricas clave + gap + LR en un solo PNG.


In [ ]:
fig = plt.figure(figsize=(16, 10), facecolor='#0f1117')
gs  = gridspec.GridSpec(3, 2, hspace=0.45, wspace=0.3)

# ── (0,0) Loss ──────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
ax.plot(epochs, H['train_loss'], color=C_TRAIN, label='Train')
ax.plot(epochs, H['val_loss'],   color=C_VAL,   label='Val', linestyle='--')
if TEST_LOSS: ax.axhline(TEST_LOSS, color=C_TEST, linestyle=':', lw=1.2, label=f'Test={TEST_LOSS:.3f}')
if best_epoch: ax.axvline(best_epoch, color='#ffffff15', lw=1)
ax.set_title('Loss', fontsize=10); ax.legend(fontsize=8)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

# ── (0,1) Accuracy ──────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 1])
ax.plot(epochs, H['train_acc'], color=C_TRAIN, label='Train')
ax.plot(epochs, H['val_acc'],   color=C_VAL,   label='Val',  linestyle='--')
if TEST_ACC: ax.axhline(TEST_ACC, color=C_TEST, linestyle=':', lw=1.2, label=f'Test={TEST_ACC:.3f}')
if best_epoch: ax.axvline(best_epoch, color='#ffffff15', lw=1)
ax.set_title('Accuracy', fontsize=10); ax.legend(fontsize=8)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

# ── (1,0) Overfitting gap ────────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
gap = [ta-va for ta,va in zip(H['train_acc'], H['val_acc'])]
ax.fill_between(epochs, gap, alpha=0.3, color=C_GAP)
ax.plot(epochs, gap, color=C_GAP, lw=1.5)
ax.axhline(0,    color='#ffffff22', lw=0.8)
ax.axhline(0.05, color=C_GAP, lw=0.6, linestyle='--', alpha=0.5)
ax.set_title('Acc Gap (train−val)', fontsize=10)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

# ── (1,1) F1 / Precision / Recall ────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
ax.plot(epochs, H['f1'],        color=C_F1,     label='F1',        lw=2)
ax.plot(epochs, H['precision'], color=C_PREC,   label='Precision', lw=1.5, ls='--')
ax.plot(epochs, H['recall'],    color=C_RECALL, label='Recall',    lw=1.5, ls=':')
if TEST_F1: ax.axhline(TEST_F1, color=C_TEST, linestyle=':', lw=1.2, label=f'Test F1={TEST_F1:.3f}')
ax.set_title('F1 / Precision / Recall (val)', fontsize=10); ax.legend(fontsize=8)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

# ── (2,0) LR ─────────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 0])
ax.semilogy(epochs, H['lr'], color=C_LR, lw=2)
for i in lr_drops:
    ax.axvline(epochs[i], color='#ffffff15', lw=1)
ax.set_title('Learning Rate (log)', fontsize=10)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

# ── (2,1) Texto de diagnóstico ───────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 1])
ax.axis('off')

max_gap   = max(gap)
final_ep  = epochs[-1]
val_trend = H['val_acc'][-1] - H['val_acc'][max(0, len(H['val_acc'])-3)]

lines_diag = [
    f"Épocas entrenadas : {final_ep}",
    f"Mejor val_acc     : {best_acc:.4f}  (epoch {best_epoch})",
    f"Test acc          : {TEST_ACC if TEST_ACC else 'N/A'}",
    "",
    f"Acc gap máx       : {max_gap:.4f}  {'⚠️' if max_gap>0.05 else '✅'}",
    f"LR drops          : {len(lr_drops)}",
    f"Tendencia val (ult 3 ep): {val_trend:+.4f}  {'📈' if val_trend>0 else '📉' if val_trend<0 else '➡️'}",
    "",
    "── Señales de overfitting ──",
    f"  acc_gap > 5%   : {'SÍ ⚠️' if max_gap>0.05 else 'NO ✅'}",
    f"  val plateau    : {'SÍ ⚠️' if abs(val_trend)<0.002 else 'NO ✅'}",
]

for i, line in enumerate(lines_diag):
    color = '#ef9a9a' if '⚠️' in line else '#aed581' if '✅' in line else '#c8d0e0'
    ax.text(0.05, 0.95 - i*0.075, line, transform=ax.transAxes,
            fontsize=8.5, color=color, va='top', fontfamily='monospace')

fig.suptitle('Training Diagnostics — HybridCNN', fontsize=14,
             fontweight='bold', color='#e8eaf0', y=1.01)

plt.savefig('00_dashboard.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('💾 Guardado: 00_dashboard.png')


## 10 · Diagnóstico automático

Conclusiones accionables basadas en las métricas del historial.


In [ ]:
print('=' * 60)
print('🔍 DIAGNÓSTICO AUTOMÁTICO')
print('=' * 60)

issues = []
suggestions = []

# 1. Gap acc
max_gap = max(ta-va for ta,va in zip(H['train_acc'], H['val_acc']))
final_gap = H['train_acc'][-1] - H['val_acc'][-1]
if final_gap > 0.07:
    issues.append(f'Overfitting severo: acc_gap final = {final_gap:.3f}')
    suggestions.append('→ Subir dropout (0.35–0.4), más weight_decay, o early stopping más agresivo')
elif final_gap > 0.04:
    issues.append(f'Overfitting moderado: acc_gap final = {final_gap:.3f}')
    suggestions.append('→ Revisar si el val set está contaminado con muestras de train')

# 2. Tendencia de val en últimas épocas
n = min(3, len(H['val_acc']))
val_trend = H['val_acc'][-1] - H['val_acc'][-n]
if val_trend < -0.005:
    issues.append(f'Val acc decreciente en las últimas {n} épocas (Δ={val_trend:.4f})')
    suggestions.append('→ El modelo ya pasó su pico — usar el checkpoint del best_epoch')
elif abs(val_trend) < 0.002:
    issues.append(f'Val acc estancada en las últimas {n} épocas (Δ={val_trend:.4f})')
    suggestions.append('→ LR demasiado bajo o capacidad saturada — probar cosine annealing')

# 3. LR drops sin mejora
for drop_idx in lr_drops:
    post = min(drop_idx + 2, len(H['val_acc']) - 1)
    improvement = H['val_acc'][post] - H['val_acc'][drop_idx]
    if improvement < 0.002:
        issues.append(f'LR drop en epoch {epochs[drop_idx]} sin mejora posterior (Δacc={improvement:.4f})')
        suggestions.append('→ El modelo estaba ya sobreajustado antes del drop')

# 4. acc vs f1 gap
avg_af_gap = np.mean([a-f for a,f in zip(H['val_acc'], H['f1'])])
if avg_af_gap > 0.04:
    issues.append(f'Acc >> F1 de forma sostenida (Δ medio={avg_af_gap:.3f})')
    suggestions.append('→ Clases desbalanceadas — revisar distribución train vs test y usar WeightedRandomSampler')

# 5. Gap val→test
if TEST_ACC is not None:
    vt_gap = best_acc - TEST_ACC
    if vt_gap > 0.06:
        issues.append(f'Gap val→test grande: {vt_gap:.3f}')
        suggestions.append('→ Distribución de test distinta a val/train (data leakage en split o fuentes distintas)')

# Output
if not issues:
    print('✅ No se detectaron problemas graves.')
else:
    for issue, sug in zip(issues, suggestions):
        print(f'\n⚠️  {issue}')
        print(f'   {sug}')

print('\n' + '=' * 60)
print(f'📌 Usar siempre checkpoint epoch={best_epoch} (best_val_acc={best_acc:.4f})')
if TEST_ACC:
    print(f'📌 Gap val→test: {best_acc-TEST_ACC:.4f}  |  test_acc={TEST_ACC:.4f}')
